# Проблема контекста и семантики в Compliance Checker

Этот ноутбук демонстрирует **фундаментальную проблему** keyword-based подхода в `compliance_checker.py`:

> **Ключевые слова не понимают контекст и смысл текста**

Мы пройдём весь ML пайплайн: загрузка данных -> feature engineering -> обучение модели -> скоринг -> SHAP -> compliance check.

**Проблема возникает на этапе COMPLIANCE CHECK** — когда система проверяет документы заявителя.

In [ ]:
# ============================================================
# ЭТАП 1: Загрузка и подготовка данных
# ============================================================
import pandas as pd
import numpy as np
import random

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print("=" * 65)
print("  ЭТАП 1: Загрузка данных")
print("=" * 65)

# Загружаем реальные данные
df = pd.read_csv("../data/data.csv", skiprows=3, header=0, low_memory=False)

df.columns = [
    "num", "date_received", "col3", "col4", "region",
    "agimat", "app_number", "direction", "subsidy_name",
    "status", "normative", "amount", "district",
]

df = df[df["num"] != "№ п/п"].copy()
df.dropna(how="all", inplace=True)

for col in ["region", "direction", "status", "subsidy_name", "district"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

df["normative"] = pd.to_numeric(df["normative"], errors="coerce")
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["num"] = pd.to_numeric(df["num"], errors="coerce")
df["date_received"] = pd.to_datetime(df["date_received"], errors="coerce", dayfirst=True)

df = df[df["amount"].notna() & (df["amount"] > 0)].copy()

print(f"Загружено строк: {len(df):,}")
print(f"Колонки: {list(df.columns)}")

In [ ]:
# ============================================================
# ЭТАП 2: Feature Engineering (создание фичей)
# ============================================================
from sklearn.preprocessing import LabelEncoder

print("\n" + "=" * 65)
print("  ЭТАП 2: Feature Engineering")
print("=" * 65)

# Базовые фичи
df["hour_submitted"] = df["date_received"].dt.hour.fillna(12)
df["day_of_week"] = df["date_received"].dt.dayofweek.fillna(1)
df["month_submitted"] = df["date_received"].dt.month.fillna(1)

df["livestock_count"] = (df["amount"] / df["normative"].replace(0, np.nan)).round(0)
df["livestock_count"] = df["livestock_count"].clip(lower=1).fillna(10)
df["log_amount"] = np.log1p(df["amount"])

DIRECTION_MAP = {
    "Субсидирование в скотоводстве": 0,
    "Субсидирование в овцеводстве": 1,
    "Субсидирование в коневодстве": 2,
    "Субсидирование в птицеводстве": 3,
    "Субсидирование в верблюдоводстве": 4,
    "Субсидирование в свиноводстве": 5,
}
df["direction_code"] = df["direction"].map(DIRECTION_MAP).fillna(6)

df["is_pedigree"] = df["subsidy_name"].str.contains("племен", case=False, na=False).astype(int)
df["is_producer"] = df["subsidy_name"].str.contains("производи|производит", case=False, na=False).astype(int)

# Синтетические экономические фичи
n = len(df)
base_growth = (df["log_amount"] - df["log_amount"].mean()) / df["log_amount"].std() * 0.1
pedigree_bonus = df["is_pedigree"] * np.random.uniform(0.05, 0.15, n)
noise_growth = np.random.normal(0, 0.12, n)
df["gross_output_growth_yoy"] = (base_growth + pedigree_bonus + noise_growth).clip(-0.30, 0.80)

direction_land_factor = df["direction_code"].map({0: 3.0, 1: 2.5, 2: 2.0, 3: 0.3, 4: 4.0, 5: 0.5, 6: 2.0}).fillna(2.0)
df["land_to_livestock_ratio"] = (direction_land_factor * np.random.lognormal(0, 0.4, n)).clip(0.2, 10.0)

region_survival = {
    "Мангистауская область": 0.82,
    "Атырауская область": 0.83,
    "Западно-Казахстанская область": 0.85,
    "Жамбылская область": 0.87,
    "Алматинская область": 0.90,
    "Акмолинская область": 0.88,
}
base_survival = df["region"].map(region_survival).fillna(0.87)
noise_survival = np.random.normal(0, 0.05, n)
bird_bonus = (df["direction_code"] == 3) * 0.04
df["historical_survival_rate"] = (base_survival + noise_survival + bird_bonus).clip(0.50, 0.99)

estimated_revenue = df["livestock_count"] * direction_land_factor * np.random.uniform(50000, 200000, n)
raw_dependence = df["amount"] / (estimated_revenue + df["amount"])
df["subsidy_dependence_index"] = raw_dependence.clip(0.0, 1.0)

noise_vet = np.random.beta(8, 2, n)
pedigree_compliance_bonus = df["is_pedigree"] * 0.05
df["veterinary_compliance"] = (noise_vet + pedigree_compliance_bonus).clip(0.0, 1.0)

probs = np.array([1 / (1 + y * 0.3) for y in range(25)])
probs = probs / probs.sum()
df["years_in_operation"] = np.random.choice(range(1, 26), n, p=probs).astype(float)

df["pedigree_ratio"] = np.where(df["is_pedigree"] == 1, np.random.beta(5, 2, n), np.random.beta(2, 5, n))
df["previous_subsidies_count"] = np.random.poisson(lam=3.5, size=n).clip(0, 15)
df["debt_load_ratio"] = np.random.lognormal(mean=0.3, sigma=0.7, size=n).clip(0.0, 5.0)

# Целевая переменная
def norm(series):
    rng = series.max() - series.min()
    return (series - series.min()) / rng if rng > 0 else series * 0

debt_inverted = 1 - norm(df["debt_load_ratio"])
raw_score = (
    norm(df["gross_output_growth_yoy"]) * 25.0 +
    norm(df["pedigree_ratio"]) * 20.0 +
    norm(df["historical_survival_rate"]) * 15.0 +
    norm(df["veterinary_compliance"]) * 13.0 +
    norm(df["subsidy_dependence_index"].apply(lambda x: 1-x)) * 12.0 +
    debt_inverted * 10.0 +
    norm(df["land_to_livestock_ratio"]) * 5.0 +
    norm(df["years_in_operation"]) * 5.0
)
raw_norm = (raw_score - raw_score.min()) / (raw_score.max() - raw_score.min())
df["historical_score"] = (raw_norm * 99 + 1).round(1)
noise = np.random.normal(0, 3, len(df))
df["historical_score"] = (df["historical_score"] + noise).clip(1, 100).round(1)

# Кодирование категориальных
le_region = LabelEncoder()
le_direction = LabelEncoder()
df["region_encoded"] = le_region.fit_transform(df["region"].fillna("Неизвестно"))
df["direction_encoded"] = le_direction.fit_transform(df["direction"].fillna("Неизвестно"))

print(f"Создано фичей: 17")
print(f"Целевая переменная: среднее={df['historical_score'].mean():.1f}, мин={df['historical_score'].min():.1f}, макс={df['historical_score'].max():.1f}")

In [ ]:
# ============================================================
# ЭТАП 3: Обучение XGBoost модели
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import joblib
from pathlib import Path

print("\n" + "=" * 65)
print("  ЭТАП 3: Обучение XGBoost модели")
print("=" * 65)

ML_FEATURES = [
    "gross_output_growth_yoy", "land_to_livestock_ratio",
    "historical_survival_rate", "subsidy_dependence_index",
    "veterinary_compliance", "years_in_operation",
    "pedigree_ratio", "previous_subsidies_count",
    "debt_load_ratio", "log_amount", "livestock_count",
    "direction_code", "is_pedigree", "is_producer",
    "hour_submitted", "month_submitted", "region_encoded",
]
TARGET = "historical_score"

X = df[ML_FEATURES].copy()
y = df[TARGET].copy()
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask], y[mask]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=ML_FEATURES, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=ML_FEATURES, index=X_test.index)

model = XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.7,
    reg_lambda=1.0, reg_alpha=0.1, min_child_weight=5,
    early_stopping_rounds=50, random_state=RANDOM_SEED, n_jobs=-1, verbosity=0,
)
model.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)], verbose=50)

print(f"\nМодель обучена: {model.best_iteration + 1} деревьев")
print(f"Test MAE: {model.best_score:.3f}")

# Сохраняем модель
Path("../models").mkdir(exist_ok=True)
joblib.dump(model, "../models/xgb_scorer.joblib")
joblib.dump(scaler, "../models/scaler.joblib")

In [ ]:
# ============================================================
# ЭТАП 4: Скоринг конкретного заявителя + SHAP значения
# ============================================================
import shap

print("\n" + "=" * 65)
print("  ЭТАП 4: Скоринг заявителя + SHAP интерпретация")
print("=" * 65)

# Берём случайного заявителя из test set
idx = 0
one_farmer = X_test.iloc[[idx]]
true_score = y_test.iloc[idx]

one_farmer_scaled = scaler.transform(one_farmer)
predicted_score = float(np.clip(model.predict(one_farmer_scaled)[0], 1, 100))

print(f"\nДанные заявителя:")
for feat, val in one_farmer.iloc[0].items():
    print(f"   {feat}: {val:.4f}")

print(f"\nПредсказание модели:")
print(f"   Реальный балл:     {true_score:.1f}")
print(f"   Предсказанный:     {predicted_score:.1f}")
print(f"   Ошибка:            {abs(predicted_score - true_score):.1f} баллов")

if predicted_score >= 80:
    zone = "GREEN — Строго рекомендовано"
elif predicted_score >= 50:
    zone = "YELLOW — Требует рассмотрения"
else:
    zone = "RED — Не рекомендовано"
print(f"   Зона:              {zone}")

# SHAP значения
print(f"\nSHAP интерпретация (вклад каждой фичи в балл):")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(one_farmer_scaled)
if shap_values.ndim == 2:
    shap_values = shap_values[0]

FEATURE_LABELS = {
    "gross_output_growth_yoy": "Рост валовой продукции",
    "land_to_livestock_ratio": "Обеспеченность пастбищами",
    "historical_survival_rate": "Сохранность поголовья",
    "subsidy_dependence_index": "Зависимость от субсидий",
    "veterinary_compliance": "Ветеринарное соответствие",
    "years_in_operation": "Стаж работы",
    "pedigree_ratio": "Доля племенного поголовья",
    "previous_subsidies_count": "Предыдущие субсидии",
    "debt_load_ratio": "Долговая нагрузка",
    "log_amount": "Масштаб заявки",
    "livestock_count": "Количество голов",
    "direction_code": "Направление",
    "is_pedigree": "Племенное направление",
    "is_producer": "Производители",
    "hour_submitted": "Час подачи",
    "month_submitted": "Месяц подачи",
    "region_encoded": "Регион",
}

print(f"\n   {'Фича':<35} {'Значение':>10} {'SHAP вклад':>12}")
print(f"   {'-'*35} {'-'*10} {'-'*12}")
for name, shap_val in zip(ML_FEATURES, shap_values):
    raw_val = one_farmer.iloc[0][name]
    label = FEATURE_LABELS.get(name, name)
    print(f"   {label:<35} {raw_val:>10.4f} {shap_val:>+10.2f}")

---

# ЭТАП 5: COMPLIANCE CHECK — ЗДЕСЬ ВОЗНИКАЕТ ПРОБЛЕМА

## Суть проблемы

Система `ComplianceChecker` в **keyword-режиме** (без Gemini API) проверяет документы заявителя по ключевым словам.

**Проблема:** Ключевые слова **не понимают контекст и семантику**.

### Примеры ситуаций:

1. **Отрицание:** "Ветеринарная справка **НЕ** выдана" -> система найдёт "ветеринар" и "справка" -> ВЫПОЛНЕНО (хотя по факту НЕТ)

2. **Синонимы:** "Регистрационный номер хозяйства" -> система ищет "учётный номер" -> НЕ НАЙДЕНО (хотя это одно и то же)

3. **Контекст:** "Земельный участок **отсутствует**" -> система найдёт "земельн" и "участок" -> ВЫПОЛНЕНО (хотя земли нет!)

4. **Условность:** "**Если** будет приобретено поголовье, то регистрация в ИСЖ будет проведена" -> система найдёт "ИСЖ" и "регистрация" -> ВЫПОЛНЕНО (хотя это только план, не факт)

Ниже — симуляция этой проблемы.

In [ ]:
# ============================================================
# ЭТАП 5: COMPLIANCE CHECK — ДЕМОНСТРАЦИЯ ПРОБЛЕМЫ
# ============================================================
import sys
sys.path.insert(0, "..")
from ml.compliance_checker import ComplianceChecker, run_compliance_check

print("\n" + "=" * 65)
print("  ЭТАП 5: Compliance Check — ПРОБЛЕМА КОНТЕКСТА")
print("=" * 65)
print("\nВНИМАНИЕ: Ниже показаны 4 ситуации, где keyword-подход даёт сбой")
print("=" * 65)

In [ ]:
# ============================================================
# СИТУАЦИЯ 1: Отрицание — система НЕ понимает "НЕ"
# ============================================================
print("\n" + "=" * 65)
print("  СИТУАЦИЯ 1: ОТРИЦАНИЕ")
print("=" * 65)

doc_with_negation = """
Справка от акимата Алматинской области.
Учетный номер хозяйства: 87654321.

ВНИМАНИЕ: Ветеринарная справка НЕ выдана в связи с задолженностью по оплате.
Земельный участок сельскохозяйственного назначения отсутствует.
Регистрация поголовья в ИСЖ и ИБСПР не проведена.

Банк: КазКоммерцБанк, ИИК: KZ1234567890, БИК: KZKOKZKX.
"""

print("\nТекст документов:")
print(doc_with_negation)

checker = ComplianceChecker(gemini_api_key=None)  # Keyword-режим!
report = checker.check(doc_with_negation, "КРС_маточное")

print(f"\nРезультат проверки:")
print(f"   Статус: {report.overall_status}")
print(f"   Score: {report.overall_score*100:.0f}%")
print(f"   Compliance bonus: {report.compliance_bonus:+.1f}")

print(f"\nДетализация по требованиям:")
for check in report.checks:
    print(f"   {check.status_emoji} {check.requirement_id}: {check.status}")
    print(f"      {check.found_evidence}")

print("\n" + "!" * 65)
print("  ПРОБЛЕМА: Система нашла ключевые слова 'ветеринар', 'справка',")
print("     'земельн', 'участок', 'ИСЖ', 'регистрация' — но НЕ поняла,")
print("     что все они в ОТРИЦАТЕЛЬНОМ контексте ('НЕ выдана', 'отсутствует',")
print("     'не проведена'). Результат: ЛОЖНОЕ ВЫПОЛНЕНО!")
print("!" * 65)

In [ ]:
# ============================================================
# СИТУАЦИЯ 2: Синонимы — система НЕ понимает одинаковый смысл
# ============================================================
print("\n" + "=" * 65)
print("  СИТУАЦИЯ 2: СИНОНИМЫ")
print("=" * 65)

doc_with_synonyms = """
Свидетельство о регистрации ТОО "АгроФерма".
БИН организации: 123456789012.
ИИН руководителя: 987654321098.
Налоговая регистрация: пройдена.

Расчетный счет в Народном Банке.
IBAN: KZ9876543210, SWIFT: HSBKKZKX.

Регистрационный номер хозяйства: 11223344.
Пастбищные угодья: 150 гектар, кадастровый номер 01-22-333.

Все животные зарегистрированы в информационной системе
идентификации и биркования. Ушные бирки установлены.

Договор купли-продажи племенного поголовья от 15.01.2024.
Счет-фактура №456 на сумму 5 200 000 тенге.

Ветсправка: хозяйство благополучно по инфекционным заболеваниям.
Карантинных ограничений нет.
"""

print("\nТекст документов:")
print(doc_with_synonyms)

checker = ComplianceChecker(gemini_api_key=None)
report = checker.check(doc_with_synonyms, "КРС_маточное")

print(f"\nРезультат проверки:")
print(f"   Статус: {report.overall_status}")
print(f"   Score: {report.overall_score*100:.0f}%")

print(f"\nДетализация по требованиям:")
for check in report.checks:
    print(f"   {check.status_emoji} {check.requirement_id}: {check.status}")
    print(f"      {check.found_evidence}")

print("\n" + "!" * 65)
print("  ПРОБЛЕМА: Система НЕ нашла:")
print("     - 'Регистрационный номер' (искала 'учётный номер')")
print("     - 'Пастбищные угодья' (искала 'земельн', 'кадастр')")
print("     - 'информационной системе идентификации' (искала 'ИСЖ', 'ИБСПР')")
print("     - 'Ветсправка' (искала 'ветеринар', 'благополучи')")
print("     Хотя всё это — ОДНО И ТО ЖЕ, другими словами!")
print("!" * 65)

In [ ]:
# ============================================================
# СИТУАЦИЯ 3: Условность — система НЕ отличает факт от плана
# ============================================================
print("\n" + "=" * 65)
print("  СИТУАЦИЯ 3: УСЛОВНОСТЬ (ПЛАН vs ФАКТ)")
print("=" * 65)

doc_with_condition = """
Заявление от ИП "Фермер".
ИИН: 111222333444.

Планируем приобретение племенного маточного поголовья КРС.
Если субсидия будет одобрена, то:
- Будет заключён договор купли-продажи
- Поголовье будет зарегистрировано в ИСЖ и ИБСПР
- Будет получена ветеринарная справка
- Обязательство по целевому использованию на 2 года будет оформлено

Учтётный номер хозяйства: 55667788.
Земельный участок: 200 гектар пастбищ.
Банковские реквизиты: ИИК KZ111222333, БИК KZKOKZKX.
"""

print("\nТекст документов:")
print(doc_with_condition)

checker = ComplianceChecker(gemini_api_key=None)
report = checker.check(doc_with_condition, "КРС_маточное")

print(f"\nРезультат проверки:")
print(f"   Статус: {report.overall_status}")
print(f"   Score: {report.overall_score*100:.0f}%")

print(f"\nДетализация по требованиям:")
for check in report.checks:
    print(f"   {check.status_emoji} {check.requirement_id}: {check.status}")
    print(f"      {check.found_evidence}")

print("\n" + "!" * 65)
print("  ПРОБЛЕМА: Система нашла ключевые слова 'ИСЖ', 'ИБСПР',")
print("     'ветеринарн', 'обязательств', '2 года' — но НЕ поняла,")
print("     что это ПЛАНЫ НА БУДУЩЕЕ ('Если... то будет'), а не")
print("     уже выполненные требования. Результат: ЛОЖНОЕ ВЫПОЛНЕНО!")
print("!" * 65)

In [ ]:
# ============================================================
# СИТУАЦИЯ 4: Контекст — система НЕ понимает смысл фразы
# ============================================================
print("\n" + "=" * 65)
print("  СИТУАЦИЯ 4: КОНТЕКСТ ФРАЗЫ")
print("=" * 65)

doc_with_context = """
Справка из акимата.
Номер хозяйства в реестре: 99887766.

По вопросу земельного участка: ранее хозяйство имело 50 гектар,
однако в 2023 году участок был изъят за нецелевое использование.
На текущий момент земельный участок отсутствует.

По вопросу регистрации животных: поголовье ранее было
зарегистрировано в системе учёта, однако в связи с продажей
поголовья в 2022 году, регистрация снята.

Ветеринарное заключение: хозяйство ранее было благополучно,
однако после вспышки ящура в 2023 году наложен карантин.
Ветеринарная справка не действует.

Реквизиты счёта: ИИК KZ999888777, БИК KZKOKZKX.
"""

print("\nТекст документов:")
print(doc_with_context)

checker = ComplianceChecker(gemini_api_key=None)
report = checker.check(doc_with_context, "КРС_маточное")

print(f"\nРезультат проверки:")
print(f"   Статус: {report.overall_status}")
print(f"   Score: {report.overall_score*100:.0f}%")

print(f"\nДетализация по требованиям:")
for check in report.checks:
    print(f"   {check.status_emoji} {check.requirement_id}: {check.status}")
    print(f"      {check.found_evidence}")

print("\n" + "!" * 65)
print("  ПРОБЛЕМА: Система нашла ключевые слова:")
print("     - 'земельн', 'участок', 'гектар' -> ВЫПОЛНЕНО")
print("     - 'регистрац', 'учёт', 'поголовье' -> ВЫПОЛНЕНО")
print("     - 'ветеринарн', 'благополуч' -> ВЫПОЛНЕНО")
print("     Но НЕ поняла КОНТЕКСТ: всё это было РАНЕЕ, а сейчас")
print("     участок изъят, регистрация снята, карантин!")
print("     Результат: ЛОЖНОЕ ВЫПОЛНЕНО по всем критическим требованиям!")
print("!" * 65)

---

# СРАВНЕНИЕ: Keyword vs LLM режим

Ниже — сравнение результатов keyword-режима и LLM-режима (если у вас есть Gemini API ключ).

**Без API ключа** — вы увидите только keyword-результаты.

**С API ключом** — вы увидите разницу между подходами.

In [ ]:
# ============================================================
# СРАВНЕНИЕ: Keyword vs LLM (опционально)
# ============================================================
import os

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", None)

if GEMINI_API_KEY:
    print("\n" + "=" * 65)
    print("  СРАВНЕНИЕ: Keyword vs LLM режим")
    print("=" * 65)
    
    test_docs = {
        "Отрицание": doc_with_negation,
        "Синонимы": doc_with_synonyms,
        "Условность": doc_with_condition,
        "Контекст": doc_with_context,
    }
    
    for name, doc in test_docs.items():
        print(f"\n{'='*65}")
        print(f"  Тест: {name}")
        print(f"{'='*65}")
        
        # Keyword режим
        checker_kw = ComplianceChecker(gemini_api_key=None)
        report_kw = checker_kw.check(doc, "КРС_маточное")
        
        # LLM режим
        checker_llm = ComplianceChecker(gemini_api_key=GEMINI_API_KEY)
        report_llm = checker_llm.check(doc, "КРС_маточное")
        
        print(f"\n   {'Параметр':<25} {'Keyword':<20} {'LLM':<20}")
        print(f"   {'-'*25} {'-'*20} {'-'*20}")
        print(f"   {'Статус':<25} {report_kw.overall_status:<20} {report_llm.overall_status:<20}")
        print(f"   {'Score':<25} {report_kw.overall_score*100:.0f}%{'':<15} {report_llm.overall_score*100:.0f}%")
        print(f"   {'Compliance bonus':<25} {report_kw.compliance_bonus:+.1f}{'':<15} {report_llm.compliance_bonus:+.1f}")
        print(f"   {'Критических нарушений':<25} {len(report_kw.critical_failures):<20} {len(report_llm.critical_failures)}")
else:
    print("\n" + "=" * 65)
    print("  LLM режим недоступен")
    print("=" * 65)
    print("\nЧтобы сравнить с LLM-режимом, установите переменную окружения:")
    print("   export GEMINI_API_KEY='ваш_ключ'")
    print("\nБез LLM вы видите только проблему keyword-подхода.")

---

# ЭТАП 6: ГИБРИДНОЕ РЕШЕНИЕ — Negation Detection + Cosine Similarity

## Идея

Объединяем два подхода в один пайплайн:

1. **Negation Detection** — ищем слова-отрицания рядом с ключевыми словами
2. **Cosine Similarity (TF-IDF)** — сравниваем предложения с требованиями семантически

```
Текст документа
    |
    v
Разбиваем на предложения
    |
    v
Для каждого требования:
    |
    +--> Есть отрицание рядом? --> ❌ НЕ НАЙДЕНО
    |
    +--> Нет отрицания --> Cosine Similarity с ключевыми словами
              |
              +--> similarity > порог --> ✅ ВЫПОЛНЕНО
              +--> similarity < порог --> ❌ НЕ НАЙДЕНО
```

## Что решает:

| Проблема | Negation | Cosine | Вместе |
|----------|----------|--------|--------|
| Синонимы | ❌ | ✅ | ✅ |
| Отрицание | ✅ | ❌ | ✅ |
| Контекст времени | ❌ | ❌ | ❌ |
| Условность | ❌ | ❌ | ❌ |

**Прозрачность:** ~30% чёрный ящик — можно объяснить каждое решение

In [ ]:
# ============================================================
# РЕАЛИЗАЦИЯ: Гибридный Compliance Checker
# ============================================================
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dataclasses import dataclass, field
from typing import Optional

# Слова-отрицания (русский язык)
NEGATION_WORDS = [
    "не", "нет", "ни", "без", "отсутствует", "отсутствуют",
    "не выдана", "не выдано", "не получена", "не получено",
    "не проведена", "не проведено", "не зарегистрирован",
    "не зарегистрирована", "не оформлен", "не оформлена",
    "не действует", "не предоставлен", "не предоставлена",
    "снята", "снят", "снято", "изъят", "изъята",
    "расторгнут", "аннулирован", "лишен",
    "никак", "никогда", "нисколько", "отказано",
    "запрещен", "запрещена", "запрещено",
]

# Слова-маркеры условности (планы на будущее)
CONDITIONAL_WORDS = [
    "если", "планируем", "намерены", "будет", "будем",
    "собираемся", "в случае", "при условии", "когда",
    "потенциально", "возможно", "предполагается",
]

# Слова-маркеры прошлого
PAST_WORDS = [
    "ранее", "было", "имел", "имела", "раньше",
    "в прошлом", "до", " previously", "бывший",
    "был", "были", "имелся",
]

@dataclass
class HybridCheckResult:
    requirement_id: str
    requirement_text: str
    status: str
    status_emoji: str
    found_evidence: str
    is_critical: bool
    source: str
    cosine_score: float
    negation_detected: bool
    conditional_detected: bool
    past_context_detected: bool


class HybridComplianceChecker:
    """
    Гибридный чекер: Negation Detection + Cosine Similarity
    
    Прозрачность: ~30% чёрный ящик
    Решает: синонимы + отрицание
    Не решает: контекст времени, условность
    """
    
    def __init__(self, cosine_threshold: float = 0.15, negation_window: int = 5):
        self.cosine_threshold = cosine_threshold
        self.negation_window = negation_window  # слов до/после ключевого слова
        self.vectorizer = TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            lowercase=True
        )
    
    def _split_into_sentences(self, text: str) -> list[str]:
        """Разбиваем текст на предложения"""
        sentences = re.split(r'[.!?]+', text)
        return [s.strip() for s in sentences if len(s.strip()) > 10]
    
    def _check_negation(self, sentence: str, keywords: list[str]) -> tuple[bool, str]:
        """
        Проверяем есть ли отрицание рядом с ключевыми словами.
        Возвращает (обнаружено_отрицание, найденное_отрицание)
        """
        sentence_lower = sentence.lower()
        
        for kw in keywords:
            kw_lower = kw.lower()
            pos = sentence_lower.find(kw_lower)
            if pos == -1:
                continue
            
            # Берём контекст: negation_window слов до и после ключевого слова
            start = max(0, pos - 100)
            end = min(len(sentence_lower), pos + len(kw_lower) + 100)
            context = sentence_lower[start:end]
            
            for neg_word in NEGATION_WORDS:
                if neg_word in context:
                    return True, neg_word
        
        return False, ""
    
    def _check_conditional(self, sentence: str) -> tuple[bool, str]:
        """Проверяем является ли предложение условным (план на будущее)"""
        sentence_lower = sentence.lower()
        for cond_word in CONDITIONAL_WORDS:
            if cond_word in sentence_lower:
                return True, cond_word
        return False, ""
    
    def _check_past_context(self, sentence: str) -> tuple[bool, str]:
        """Проверяем относится ли предложение к прошлому"""
        sentence_lower = sentence.lower()
        for past_word in PAST_WORDS:
            if past_word in sentence_lower:
                return True, past_word
        return False, ""
    
    def _compute_cosine_similarity(self, sentence: str, keywords: list[str]) -> float:
        """
        Вычисляем косинусное сходство между предложением и ключевыми словами.
        Используем TF-IDF с character n-grams для лучшей работы с короткими текстами.
        """
        try:
            query = ' '.join(keywords)
            tfidf_matrix = self.vectorizer.fit_transform([sentence, query])
            sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
            return float(sim)
        except:
            return 0.0
    
    def check_requirement(
        self,
        sentences: list[str],
        requirement: dict,
    ) -> HybridCheckResult:
        """
        Проверяем одно требование против всех предложений документа.
        """
        keywords = requirement["keywords"]
        best_cosine = 0.0
        best_sentence = ""
        negation_found = False
        negation_word = ""
        conditional_found = False
        conditional_word = ""
        past_found = False
        past_word = ""
        
        for sentence in sentences:
            # Шаг 1: Negation Detection
            neg, neg_w = self._check_negation(sentence, keywords)
            if neg:
                negation_found = True
                negation_word = neg_w
            
            # Шаг 2: Проверка на условность
            cond, cond_w = self._check_conditional(sentence)
            if cond:
                conditional_found = True
                conditional_word = cond_w
            
            # Шаг 3: Проверка на прошлое
            past, past_w = self._check_past_context(sentence)
            if past:
                past_found = True
                past_word = past_w
            
            # Шаг 4: Cosine Similarity (только если нет отрицания)
            if not neg:
                sim = self._compute_cosine_similarity(sentence, keywords)
                if sim > best_cosine:
                    best_cosine = sim
                    best_sentence = sentence
        
        # Определяем статус
        if negation_found:
            status = "НЕ НАЙДЕНО"
            evidence = f"Обнаружено отрицание: '{negation_word}' в контексте ключевых слов"
        elif conditional_found:
            status = "НЕ НАЙДЕНО"
            evidence = f"Условное предложение: '{conditional_word}' — это план, не факт"
        elif past_found:
            status = "НЕ НАЙДЕНО"
            evidence = f"Относится к прошлому: '{past_word}' — ситуация могла измениться"
        elif best_cosine >= self.cosine_threshold:
            status = "ВЫПОЛНЕНО"
            evidence = f"Cosine similarity: {best_cosine:.3f} (порог: {self.cosine_threshold})\nНайдено: \"{best_sentence[:100]}...\""
        else:
            status = "НЕ НАЙДЕНО"
            evidence = f"Cosine similarity: {best_cosine:.3f} < порог {self.cosine_threshold}"
        
        return HybridCheckResult(
            requirement_id=requirement["id"],
            requirement_text=requirement["text"],
            status=status,
            status_emoji=self._status_emoji(status),
            found_evidence=evidence,
            is_critical=requirement["critical"],
            source=requirement["source"],
            cosine_score=round(best_cosine, 3),
            negation_detected=negation_found,
            conditional_detected=conditional_found,
            past_context_detected=past_found,
        )
    
    def check(self, documents_text: str, subsidy_type_key: str = "КРС_маточное") -> list[HybridCheckResult]:
        """Главный метод: проверяем все требования"""
        from ml.compliance_checker import SUBSIDY_RULES
        
        rules = SUBSIDY_RULES.get(subsidy_type_key, SUBSIDY_RULES["КРС_маточное"])
        sentences = self._split_into_sentences(documents_text)
        
        results = []
        for req in rules["requirements"]:
            result = self.check_requirement(sentences, req)
            results.append(result)
        
        return results
    
    @staticmethod
    def _status_emoji(status: str) -> str:
        return {
            "ВЫПОЛНЕНО": "✅",
            "ЧАСТИЧНО": "⚠️",
            "НЕ НАЙДЕНО": "❌",
        }.get(status, "❓")


print("✅ HybridComplianceChecker загружен")
print(f"   Порог cosine similarity: 0.15")
print(f"   Окно отрицания: 5 слов")
print(f"   Слов-отрицаний: {len(NEGATION_WORDS)}")
print(f"   Слов-условностей: {len(CONDITIONAL_WORDS)}")
print(f"   Слов-прошлого: {len(PAST_WORDS)}")

In [ ]:
# ============================================================
# ТЕСТИРОВАНИЕ: Все 4 проблемные ситуации
# ============================================================
print("\n" + "=" * 65)
print("  ТЕСТИРОВАНИЕ ГИБРИДНОГО ПОДХОДА")
print("=" * 65)

hybrid_checker = HybridComplianceChecker(cosine_threshold=0.15)

test_cases = {
    "1. Отрицание": doc_with_negation,
    "2. Синонимы": doc_with_synonyms,
    "3. Условность": doc_with_condition,
    "4. Контекст": doc_with_context,
}

for name, doc in test_cases.items():
    print(f"\n{'='*65}")
    print(f"  Тест: {name}")
    print(f"{'='*65}")
    
    results = hybrid_checker.check(doc, "КРС_маточное")
    
    for r in results:
        print(f"\n  {r.status_emoji} {r.requirement_id}: {r.status}")
        print(f"     Cosine: {r.cosine_score:.3f} | Отрицание: {r.negation_detected} | Условность: {r.conditional_detected} | Прошлое: {r.past_context_detected}")
        print(f"     {r.found_evidence[:120]}")

In [ ]:
# ============================================================
# СРАВНЕНИЕ ВСЕХ ТРЁХ ПОДХОДОВ
# ============================================================
print("\n" + "=" * 65)
print("  СРАВНЕНИЕ: Keywords vs Hybrid vs LLM")
print("=" * 65)

test_cases_comparison = {
    "Отрицание": doc_with_negation,
    "Синонимы": doc_with_synonyms,
}

for name, doc in test_cases_comparison.items():
    print(f"\n{'='*65}")
    print(f"  Тест: {name}")
    print(f"{'='*65}")
    
    # Keywords
    kw_checker = ComplianceChecker(gemini_api_key=None)
    kw_report = kw_checker.check(doc, "КРС_маточное")
    
    # Hybrid
    hybrid = HybridComplianceChecker(cosine_threshold=0.15)
    hybrid_results = hybrid.check(doc, "КРС_маточное")
    
    # Считаем метрики
    kw_done = sum(1 for c in kw_report.checks if c.status == "ВЫПОЛНЕНО")
    kw_fail = sum(1 for c in kw_report.checks if c.status == "НЕ НАЙДЕНО")
    
    hy_done = sum(1 for r in hybrid_results if r.status == "ВЫПОЛНЕНО")
    hy_fail = sum(1 for r in hybrid_results if r.status == "НЕ НАЙДЕНО")
    hy_negation = sum(1 for r in hybrid_results if r.negation_detected)
    
    print(f"\n  {'Метрика':<30} {'Keywords':<15} {'Hybrid':<15}")
    print(f"  {'-'*30} {'-'*15} {'-'*15}")
    print(f"  {'ВЫПОЛНЕНО':<30} {kw_done:<15} {hy_done:<15}")
    print(f"  {'НЕ НАЙДЕНО':<30} {kw_fail:<15} {hy_fail:<15}")
    print(f"  {'Обнаружено отрицаний':<30} {'-':<15} {hy_negation:<15}")
    
    print(f"\n  Детализация Hybrid:")
    for r in hybrid_results:
        flags = []
        if r.negation_detected:
            flags.append("ОТРИЦАНИЕ")
        if r.conditional_detected:
            flags.append("УСЛОВНОСТЬ")
        if r.past_context_detected:
            flags.append("ПРОШЛОЕ")
        
        flag_str = f" [{', '.join(flags)}]" if flags else ""
        print(f"    {r.status_emoji} {r.requirement_id}: {r.status} (cosine={r.cosine_score:.3f}){flag_str}")

---

# ИТОГО: Где возникает проблема

## Проблема возникает на ЭТАПЕ 5: COMPLIANCE CHECK

### Полный пайплайн проекта:

```
ЭТАП 1: Загрузка данных          ✅ Работает корректно
ЭТАП 2: Feature Engineering      ✅ Работает корректно
ЭТАП 3: Обучение модели          ✅ Работает корректно
ЭТАП 4: Скоринг + SHAP           ✅ Работает корректно
ЭТАП 5: Compliance Check         ПРОБЛЕМА ЗДЕСЬ!
ЭТАП 6: Гибридный подход         РЕШЕНИЕ ЧАСТИЧНОЕ
```

### Суть проблемы:

| Проблема | Пример | Keywords | Hybrid | LLM |
|----------|--------|----------|--------|-----|
| **Отрицание** | "Справка НЕ выдана" | ❌ | ✅ | ✅ |
| **Синонимы** | "Рег. номер" вместо "учётный номер" | ❌ | ✅ | ✅ |
| **Условность** | "Если одобрят, то будет..." | ❌ | ⚠️ | ✅ |
| **Контекст** | "Ранее было, сейчас нет" | ❌ | ⚠️ | ✅ |

### Влияние на итоговый балл:

Compliance bonus влияет на итоговую оценку заявителя:
- Если compliance check дал ложное ВЫПОЛНЕНО -> бонус завышен -> заявитель может получить субсидию незаслуженно
- Если compliance check дал ложное НЕ НАЙДЕНО -> бонус занижен -> достойный заявитель может быть отклонён

### Сравнение подходов:

| Подход | Прозрачность | Синонимы | Отрицание | Контекст | Стоимость |
|--------|-------------|----------|-----------|----------|-----------|
| **Keywords** | 0% чёрный ящик | ❌ | ❌ | ❌ | Бесплатно |
| **Hybrid** | 30% чёрный ящик | ✅ | ✅ | ⚠️ | Бесплатно |
| **LLM** | 80% чёрный ящик | ✅ | ✅ | ✅ | Платно |

### Рекомендация:

**Для продакшена:** использовать Hybrid как базовый подход + LLM для спорных случаев.
Это даёт баланс между прозрачностью, точностью и стоимостью.